# Finding ORCIDs for students in the Biology and Geology of Tropical Islands Class between 1992 and 2019
Given a list of student author names we can search for ORCID metadata to help match names to correct ORCIDs. The metadata we have from ORCID profiles includs: other names, employment, education, works, peer-reviews, funding and email. In order for these metadata to be helpful, the ORCID metadata data must 1) exist in the profile and 2) be open to the public. If these data are available, it is not unusual for there to be multiple matches to many names, after all, ORCIDs exist to help identify the right person with a particular name.

This notebook provides some ORCID metadata that might help find ORCIDs, i.e. unique and persistent identifiers, for students  

To run this notebook (from your drive):
1.   In Colab press Commands in the menu above, input "out" and select **Clear all outputs** to make sure there are no old outputs. In Jupyter just press **Clear all outputs** in the menu bar.
2.   Press the arrows in the cells below to run the code in the cells. All cells with arrows must be run in order to make the notebook work!
3.   The code in these cells is hidden to make the notebook easier to read (hopefully) and help you focus on results. If you are interested, press the "Show Code" links to see code.

In [ ]:
#@title Step 1. Set up the environment for running the notebook
import pandas as pd
from   tabulate import tabulate                     # makes pretty tables in many formats from dataframes
from   IPython.display import display, Markdown     # allows computation results to be displayed in markdown
import  ipywidgets as widgets                        # interactive widgets
#
# data source can be a git repository or a local directory.
#
dataSource = 'https://raw.githubusercontent.com/tedhabermann/MooreaStudentPapers/refs/heads/main/data/'  # git repository


In [ ]:
#@title Step 2. Read the data the notebook needs: the index of authors and papers and the ORCID metadata found automatically
#
# Read data from the source
#
dataDate = '20260312_16'
if dataSource.startswith('https'):
    #
    # Data and metadata related to these papers are in a public git repository defined by dataSource
    # The index of the papers (a spreadsheet) is in this git repository as a csv file
    # read the index of student papers from git repository
    #
    url = f'{dataSource}/studentFullNames.csv'
    index_df = pd.read_csv(url)
    #if not 'Authors Primary' in index_df.columns:
    #    raise ValueError(f"Expected column 'Authors Primary' not found in index dataframe from {url}")
    #
    # Read all sheets from the ORCID metadata files in the git repository
    # These are csv files spilt out from the ORCID metadata spreadsheet (xlsx) that is in the git repository
    #
    orcid_metadata_l = ['counts', 'employment', 'education', 'works', 'funding', 'email']
    orcid_dfs = {}
    for sheet in orcid_metadata_l:
        url = f"{dataSource}/orcidMetadata_{sheet}__{dataDate}.csv"
        orcid_dfs[sheet] = pd.read_csv(url)
else:
    #
    # data for this analysis are in local files (not for use in collab)
    # Read the index of student papers from excel, which includes author names and publication years, into a dataframe
    index_file = '/Users/metadatagamechanger/ProjectsAndPlans/FieldStations/MooreaStudentPapers/index_to_moorea_class_projects_1992-2022.xlsx'
    index_df = pd.read_excel(index_file, sheet_name='MooreaClass-Export.txt', engine='openpyxl')
    #
    # Read all sheets from the ORCID metadata file (xlsx) into a dictionary of dataframes
    #
    orcid_metadata_file = f'/Users/metadatagamechanger/Repositories/MooreaStudentPapers/data/orcidMetadata__{dataDate}.xlsx'
    orcid_dfs = pd.read_excel(orcid_metadata_file, sheet_name=None)
#
# Display the input data to verify they were read correctly
print(f"{len(index_df)} rows in the index dataframe")           # Display the index count
print(f"Sheets found: {list(orcid_dfs.keys())} ")

## Match Counts
An ORCID search for a name can return any number of matches, i.e. one for each person with that name that has an ORCID. The difficulty of picking the right ORCID increases as the number of matches increases. The counts dataframe in the ORCID output has the input names in a column named 'input'. The number of times these names occur is the number of matches for the name. Add these counts to the counts dataframe as the matchCounts column.

In [ ]:
#@title Step 3. Compare the ORCID metadata with the index of papers to count the number of matches for each input name, and add it to the data
#
# Create a series of counts of the number of matches for each input name, and add it to the dataframe
#
counts_df                = orcid_dfs['counts']                                                                                 # get the counts dataframe from the dictionary of dataframes
matchCounts              = counts_df[counts_df['orcid'] != 'Not Found']['input'].value_counts()      # a series with the number of matches for each input name
counts_df['matchCounts'] = counts_df['input'].map(matchCounts).fillna(0).astype(int)                 # add the match counts to the dataframe, filling in 0 for names with no matches
#counts_df.fillna('', inplace=True)                                                                  # replace NaN with empty string for better display
#counts_df.head(20)                                                                                  # display the first 20 rows of the dataframe with match counts
print(f"{len(counts_df)} rows in the counts dataframe with match counts added")

## Define your study year.
Each year of the class has roughly 20 students which seems like a managable number for the re-curation process. 
yzour year is defined here:

In [ ]:
#@title Step 4. Define the year that you are re-curating
studyYear = 1992                        # pick a year to focus on


In [ ]:
#@title Step 5. Select the data for your year, and display the authors with match counts less than or equal to a limit
matchCountLimit = 10                        # define the maximum number of matches to be explored (default = 10)
display(Markdown(f'##Study year{studyYear} with matches <= {matchCountLimit}**'))
studyYearIndex_df = index_df[index_df['Pub Year'] == studyYear]                    # the publication years are in the index dataframe and not in counts_df
                                                                                   # studyYearIndex is the index for papers published during the study year.
studyYearAuthor_l       = list(studyYearIndex_df['Authors, Primary'].values)            # studyYearAuthor_l is the list of authors for the selected year
authorORCIDMetadata_df  = counts_df[counts_df['input'].isin(studyYearAuthor_l)]         # get the counts dataframe for the authors in the selected year

matchCountLimit = 10                                                                        # define the maximum number of counts to be explored (default = 10)
df = authorORCIDMetadata_df[authorORCIDMetadata_df['matchCounts'] <= matchCountLimit]       # select the rows with match counts less than or equal to the limit
#df.columns = ['input', 'given', 'family', 'orcid', 'matchCounts', 'other-names', 'employment',
#            'education', 'works', 'peer-reviews', 'funding', 'emails',
#            'other_names', 'Total', 'journal']
#df.matchCounts = df.matchCounts.astype(int)
display(Markdown(f'**{len(df)} authors in {studyYear} with matches <= {matchCountLimit}**'))
display(Markdown(tabulate(df.sort_values(by=['matchCounts','input']), headers=list(df.columns), tablefmt='pipe', floatfmt='.0f', showindex=False)))


In [ ]:
#@title Step 6. Select an author to look at in more detail
author_names = df['input'].unique().tolist()                    # create a list of unique author names with >= matchCounts from the selected year
author_widget = widgets.Dropdown(                               # make a pull-down widget with the author names
    options=author_names,
    description='Select Author:',
    style={'description_width': '100px'}
)
display(author_widget)                                          # display the widget to make selection

def on_author_change(change):
    global selectedAuthor
    selectedAuthor = change['new']

author_widget.observe(on_author_change, names='value')          # set up the widget to call the on_author_change function when a new author is selected   
selectedAuthor = author_widget.value                            # get the value of the selected author from the widget   


In [ ]:
#@title Step 7. Display ORCID Metadata for that author
element_l = ['email','education', 'employment', 'funding']

input = selectedAuthor
orcid_l = list(authorORCIDMetadata_df[authorORCIDMetadata_df['input']==input]['orcid'].values)

if orcid_l[0] == 'Not Found':
    print(f'No ORCID was found for {input} from the {studyYear} class')
else:
    if len(orcid_l) == 1:
        display(Markdown(f'{input} from the {studyYear} class has 1 ORCID match: [{orcid_l[0]}](https://commons.datacite.org/orcid.org/{orcid_l[0]})'))
    else:
        print(f'{input} from the {studyYear} class has {len(orcid_l)} ORCID matchs: ', ', '.join(orcid_l))
    for element in element_l:
        data = orcid_dfs[element]
        df = data[data['orcid'].isin(orcid_l)]
        if len(df) > 0:
            df = df.fillna('')
            display(Markdown(f'**{input} from the {studyYear} class {element}: ({len(df)})**'))
            display(Markdown(tabulate(df, headers=list(df.columns), tablefmt='pipe', floatfmt='.0f', showindex=False)))
        else:
            display(Markdown(f'{input} from the {studyYear} class has no {element} metadata.'))

## Show works
Typically the number of works is large relative to the other categories so they were not displayed above... Run the next cell if you would like to see the list of works for the ORCIDs

In [ ]:
#@title Step 7. Display works metadata for that author
for element in ['works']:
    data = orcid_dfs[element]
    df = data[data['orcid'].isin(orcid_l)]
    if len(df) > 0:
        df = df.fillna('')
        display(Markdown(f'**{input} from the {studyYear} class {element}: ({len(df)})**'))
        display(Markdown(tabulate(df, headers=list(df.columns), tablefmt='pipe', floatfmt='.0f', showindex=False)))
    else:
        display(Markdown(f'{input} from the {studyYear} class has no {element} metadata.'))